In [2]:
import os
# 强制指定项目根目录，所有文件生成在这里
os.chdir(r'C:\Users\20050927bzh\ACC102-S4-RD-Profitability-Analysis')

import wrds
import pandas as pd
from sqlalchemy.exc import ProgrammingError, OperationalError

# 自动创建数据文件夹
os.makedirs('data', exist_ok=True)

def fetch_wrds_data():
    try:
        # 连接WRDS数据库
        print("Connecting to WRDS database...")
        db = wrds.Connection()
        
        # 查询2018-2023年美国上市公司数据
        query = """
        SELECT gvkey, conm, fyear, xrd, ni, at, sale, sich AS siccd, dlc, dltt
        FROM comp.funda
        WHERE fyear BETWEEN 2018 AND 2023
        AND at > 0
        AND fic = 'USA'
        AND curcd = 'USD'
        AND CONSOL = 'C'
        AND datafmt = 'STD'
        AND indfmt = 'INDL'
        """
        raw_data = db.raw_sql(query)
        db.close()
        
        # 保存原始数据
        raw_data.to_csv('data/raw_compustat_data.csv', index=False)
        print(f"✅ WRDS数据获取成功！共 {len(raw_data)} 条记录，已保存至 data/ 文件夹")
        return raw_data
    
    except OperationalError:
        print("❌ WRDS连接失败：请检查账号、密码或网络")
        exit(1)
    except ProgrammingError as e:
        print(f"❌ SQL查询错误：{e}")
        exit(1)

if __name__ == "__main__":
    fetch_wrds_data()

Connecting to WRDS database...


Enter your WRDS username [20050927bzh]: bianbian050927
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


pgpass file created at C:\Users\20050927bzh\AppData\Roaming\postgresql\pgpass.conf
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
✅ WRDS数据获取成功！共 31791 条记录，已保存至 data/ 文件夹


In [3]:
import os
os.chdir(r'C:\Users\20050927bzh\ACC102-S4-RD-Profitability-Analysis')

import pandas as pd
import numpy as np

os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)

def clean_data():
    # 读取原始数据
    try:
        df = pd.read_csv('data/raw_compustat_data.csv')
    except FileNotFoundError:
        print("❌ 未找到原始数据，请先运行 01_wrds_data_fetch.py")
        exit(1)
    
    # 数据清洗
    df = df.rename(columns={'sic': 'siccd'})
    # 剔除金融、公用事业行业
    df = df[~df['siccd'].astype(str).str.startswith(('6', '49'))]
    
    # 核心变量缺失值处理
    core_vars = ['xrd', 'ni', 'at', 'sale', 'dlc', 'dltt']
    df = df.dropna(subset=core_vars)
    df = df[df['sale'] > 0]
    
    # 1%缩尾处理，消除异常值
    def winsorize(var, lower=0.01, upper=0.99):
        return np.clip(var, var.quantile(lower), var.quantile(upper))
    
    for var in core_vars:
        df[var] = winsorize(df[var])
    
    # 构建研究变量
    df['rd_intensity'] = df['xrd'] / df['at']
    df['roa'] = df['ni'] / df['at']
    df['firm_size'] = np.log(df['at'])
    df['leverage'] = (df['dlc'] + df['dltt']) / df['at']
    
    # 滞后一期研发强度
    df = df.sort_values(['gvkey', 'fyear'])
    df['rd_intensity_lag1'] = df.groupby('gvkey')['rd_intensity'].shift(1)
    df = df.dropna(subset=['rd_intensity_lag1'])
    
    # 去重
    df = df.drop_duplicates(subset=['gvkey', 'fyear'], keep='first')
    
    # 生成虚拟变量
    df['industry'] = df['siccd'].astype(str).str[:2]
    df_dummies = pd.get_dummies(df, columns=['industry', 'fyear'], drop_first=True, dtype=int)
    df_dummies[['gvkey', 'fyear']] = df[['gvkey', 'fyear']].values
    
    # 保存清洗后数据
    df_dummies.to_csv('data/processed_data.csv', index=False)
    print(f"✅ 数据清洗完成！有效样本 {len(df_dummies)} 个，已保存至 data/processed_data.csv")
    return df_dummies

if __name__ == "__main__":
    clean_data()

✅ 数据清洗完成！有效样本 10129 个，已保存至 data/processed_data.csv


In [4]:
import os
os.chdir(r'C:\Users\20050927bzh\ACC102-S4-RD-Profitability-Analysis')

import pandas as pd
from linearmodels import PanelOLS
import numpy as np

os.makedirs('output', exist_ok=True)

def run_analysis():
    # 读取清洗后数据
    try:
        df = pd.read_csv('data/processed_data.csv')
    except FileNotFoundError:
        print("❌ 未找到清洗后数据，请先运行 02_data_cleaning.py")
        exit(1)
    
    # 设置面板数据索引
    df['gvkey'] = df['gvkey'].astype(str)
    df['fyear'] = df['fyear'].astype(int)
    df = df.set_index(['gvkey', 'fyear'])
    
    # 1. 描述性统计
    desc_vars = ['roa', 'rd_intensity', 'rd_intensity_lag1', 'firm_size', 'leverage']
    desc_stats = df[desc_vars].describe().round(4)
    desc_stats.to_csv('output/descriptive_statistics.csv')
    print("="*50)
    print("📊 描述性统计结果")
    print(desc_stats)
    
    # 2. 相关性矩阵
    corr_matrix = df[desc_vars].corr().round(4)
    corr_matrix.to_csv('output/correlation_matrix.csv')
    print("\n" + "="*50)
    print("🔗 相关性矩阵")
    print(corr_matrix)
    
    # 3. 面板回归分析
    control_vars = [col for col in df.columns if col.startswith(('industry_', 'fyear_')) or col in ['firm_size', 'leverage']]
    model = PanelOLS(df['roa'], df[['rd_intensity_lag1'] + control_vars], drop_absorbed=True)
    results = model.fit(cov_type='clustered', cluster_entity=True)
    
    # 保存回归结果
    with open('output/regression_results.txt', 'w', encoding='utf-8') as f:
        f.write(str(results))
    
    print("\n" + "="*50)
    print("📈 核心回归结果")
    print(results)
    print("\n✅ 所有分析结果已保存至 output/ 文件夹")

if __name__ == "__main__":
    run_analysis()

📊 描述性统计结果
              roa  rd_intensity  rd_intensity_lag1   firm_size    leverage
count  10129.0000    10129.0000         10129.0000  10129.0000  10129.0000
mean      -0.3012        0.1362             0.1365      6.1887      0.4048
std        1.6468        0.3332             0.3047      2.5015      1.2920
min      -72.8017        0.0000             0.0000     -0.8420      0.0000
25%       -0.2699        0.0092             0.0093      4.4716      0.0808
50%       -0.0216        0.0489             0.0494      6.3676      0.2585
75%        0.0568        0.1529             0.1539      7.9116      0.4373
max        5.0198       14.6725             8.7153     11.6521     45.7352

🔗 相关性矩阵
                      roa  rd_intensity  rd_intensity_lag1  firm_size  \
roa                1.0000       -0.4676            -0.2954     0.3200   
rd_intensity      -0.4676        1.0000             0.6270    -0.3203   
rd_intensity_lag1 -0.2954        0.6270             1.0000    -0.3254   
firm_size     

In [6]:
import os
os.chdir(r'C:\Users\20050927bzh\ACC102-S4-RD-Profitability-Analysis')

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

os.makedirs('output', exist_ok=True)

def generate_plots():
    # 读取数据
    try:
        df = pd.read_csv('data/processed_data.csv')
    except FileNotFoundError:
        print("❌ 未找到数据，请先运行 02_data_cleaning.py")
        exit(1)
    
    # 全局绘图设置
    plt.rcParams['figure.dpi'] = 300
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['axes.grid'] = False

    # ========== 图1：研发强度分位数ROA对比 ==========
    df['rd_quintile'] = pd.qcut(df['rd_intensity_lag1'], 5, labels=['Q1','Q2','Q3','Q4','Q5'])
    quintile_roa = df.groupby('rd_quintile', observed=False)['roa'].mean().reset_index()
    plt.figure(figsize=(8,5))
    sns.barplot(x='rd_quintile', y='roa', hue='rd_quintile', data=quintile_roa, palette='Blues_d', legend=False)
    plt.title('Average ROA by R&D Intensity Quintile')
    plt.xlabel('R&D Intensity Quintile')
    plt.ylabel('Average ROA')
    plt.savefig('output/01_rd_quintile_roa.png', bbox_inches='tight')
    plt.close()

    # ========== 图2：相关性热力图 ==========
    corr_vars = ['roa','rd_intensity_lag1','firm_size','leverage']
    corr = df[corr_vars].corr()
    plt.figure(figsize=(6,5))
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', square=True)
    plt.title('Correlation Heatmap')
    plt.savefig('output/02_correlation_heatmap.png', bbox_inches='tight')
    plt.close()

    # ========== 图3：年度趋势图 ==========
    yearly_trend = df.groupby('fyear')[['rd_intensity','roa']].mean().reset_index()
    fig, ax1 = plt.subplots(figsize=(8,5))
    ax1.plot(yearly_trend['fyear'], yearly_trend['rd_intensity'], marker='o', color='#1f77b4', label='R&D Intensity')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('R&D Intensity', color='#1f77b4')
    ax2 = ax1.twinx()
    ax2.plot(yearly_trend['fyear'], yearly_trend['roa'], marker='s', color='#ff7f0e', label='ROA')
    ax2.set_ylabel('ROA', color='#ff7f0e')
    plt.title('Yearly Trend of R&D and ROA')
    plt.savefig('output/03_yearly_trend.png', bbox_inches='tight')
    plt.close()

    # ========== 图4：研发强度vsROA散点拟合图 ==========
    plt.figure(figsize=(8,5))
    sns.regplot(x='rd_intensity_lag1', y='roa', data=df, scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
    plt.title('R&D Intensity vs ROA')
    plt.savefig('output/04_rd_roa_scatter.png', bbox_inches='tight')
    plt.close()

    # ========== 图5：公司规模分布直方图 ==========
    plt.figure(figsize=(8,5))
    sns.histplot(df['firm_size'], bins=30, kde=True, color='green', edgecolor='black')
    plt.title('Firm Size Distribution')
    plt.savefig('output/05_firm_size_distribution.png', bbox_inches='tight')
    plt.close()

    # ========== 图6：杠杆率分位数ROA对比 ==========
    df['lev_quartile'] = pd.qcut(df['leverage'], 4, labels=['Q1','Q2','Q3','Q4'])
    leverage_roa = df.groupby('lev_quartile', observed=False)['roa'].mean().reset_index()
    plt.figure(figsize=(8,5))
    sns.barplot(x='lev_quartile', y='roa', hue='lev_quartile', data=leverage_roa, palette='RdYlGn_r', legend=False)
    plt.title('Leverage vs ROA')
    plt.savefig('output/06_leverage_roa.png', bbox_inches='tight')
    plt.close()

    print("✅ 6张可视化图表已全部生成，保存至 output/ 文件夹")

if __name__ == "__main__":
    generate_plots()

✅ 6张可视化图表已全部生成，保存至 output/ 文件夹
